# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cpu
CUDA available: False


In [3]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen3.5-2B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen3.5-2B-Instruct-bnb-4bit',
 'max_seq_length': 2048,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'eval_size': 0.02,
 'output_dir': '/kaggle/working/qwen2b_svg_lora'}

In [4]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    "xingxm/SVGX-SFT-1M",
    "OmniSVG/MMSVG-Icon",
]

In [ ]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

: 

In [6]:
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Loading xingxm/SVGX-SFT-1M ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

SVGX_SFT_GEN_51k.json:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

SVGX_SFT_GEN_51k_encode.json:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

SVGX_SFT_vision_25k_int.json:   0%|          | 0.00/585M [00:00<?, ?B/s]

SVGX_SFT_vision_25k_int_encode.json:   0%|          | 0.00/799M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

: 

: 

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    evaluation_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
train_result

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\\s\\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

for _, row in test_df.iterrows():
    svg = generate_svg(row["prompt"])
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.